<a href="https://colab.research.google.com/github/ahmedgx1010/Chess-Engine/blob/main/Chess_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 36.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=4ce6e64a84c056024a6e6f3f1f97fef45ad898cdf920450c677f915ee00b857a
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess


In [ ]:
import chess
board = chess.Board()

In [ ]:
piece_values={
    chess.PAWN:1,
    chess.KNIGHT:3,
    chess.BISHOP:3,
    chess.ROOK:5,
    chess.QUEEN:9,
    chess.KING:0,
}

def evaluate(board):
  score = 0
  if board.is_checkmate():
    if board.turn == chess.WHITE:
      return -1000
    else:
      return 1000
  if board.is_game_over():
    return 0
  for square, piece in board.piece_map().items():
    value = piece_values[piece.piece_type]
    if piece.color == chess.WHITE:
      score += value
    else:
      score -= value
  return score

In [ ]:
def minimax(board, depth, maximizing):
  if depth == 0 or board.is_game_over():
    return evaluate(board)

  if maximizing:
    best = -float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = minimax(board, depth - 1, False)
      board.pop()
      best = max(best,score)
    return best
  else:
    best = float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = minimax(board, depth - 1, True)
      board.pop()
      best = min(best, score)
    return best

In [ ]:
def best_move(board, depth):
  maximizing = board.turn == chess.WHITE
  best_score = -float("inf") if maximizing else float("inf")
  chosen = None

  for move in board.legal_moves:
    board.push(move)
    score = minimax(board, depth - 1, not maximizing)
    board.pop()

    if maximizing and score > best_score:
      best_score, chosen = score, move
    elif not maximizing and score < best_score:
      best_score, chosen = score, move

  return chosen

In [ ]:
def alphabeta(board, depth, alpha, beta, maximizing):
  if depth == 0 or board.is_game_over():
    return evaluate(board)

  if maximizing:
    best = -float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = alphabeta(board, depth - 1, alpha, beta, False)
      board.pop()
      best = max(best, score)
      alpha = max(alpha, best)
      if alpha >= beta:
        break
    return best
  else:
    best = float("inf")
    for move in board.legal_moves:
      board.push(move)
      score = alphabeta(board, depth - 1, alpha, beta, True)
      board.pop()
      best = min(best, score)
      beta = min(beta, best)
      if alpha >= beta:
        break
    return best

In [ ]:
def best_move_alphabeta(board, depth):
  maximizing = board.turn == chess.WHITE
  best_score = -float("inf") if maximizing else float("inf")
  chosen = None

  for move in board.legal_moves:
    board.push(move)
    score = alphabeta(board, depth - 1, -float("inf"), float("inf"), not maximizing)
    board.pop()

    if maximizing and score > best_score:
      best_score, chosen = score, move
    elif not maximizing and score < best_score:
      best_score, chosen = score, move

  return chosen

In [ ]:
import time

board.reset()
for m in ["e4", "e5", "Qh5", "Nc6"]:
    board.push_san(m)

start = time.time()
print("Minimax : ", best_move(board, 3), round(time.time() - start, 2), "sec")

start = time.time()
print("alphabeta : ", best_move_alphabeta(board, 3), round(time.time() - start, 2), "sec")


Minimax :  h5f5 8.37 sec
alphabeta :  h5f5 0.29 sec


In [ ]:
engines = {
    1: ("minimax", best_move),
    2: ("alphabeta", best_move_alphabeta)
}
game = chess.Board()
settings = {"engine": 2, "depth": 3}

def show_engines():
  for number, (name, _) in engines.items():
    print(f"{number}: {name}")

def new_game(engine=2, depth=3):
  if engine not in engines:
    print("Unknown engine, Choose a number:")
    show_engines()
    return
  game.reset()
  settings["engine"] = engine
  settings["depth"] = depth
  name, _ = engines[engine]
  print(f"New game: you are white, engine is {name} at depth {depth}")
  display(game)

def my_move(move_text):
  try:
    game.push_san(move_text)
  except ValueError:
    legal = [game.san(m) for m in game.legal_moves]
    print(f"'{move_text}' is illegal. legal moves: {', ' .join(legal)}")
    return

  if not game.is_game_over():
    name, engine = engines[settings["engine"]]
    reply = engine(game, settings["depth"])
    print(f"Engine ({name}) played:", game.san(reply))
    game.push(reply)


  display(game)
  if game.is_game_over():
    print("game over:", game.result())